# Dataset Analysis; the explanatory runs

This notebook is for reading and explaining the results produced by the computational analysis notebook.

It intentionally does not scan the image directory again, calculate image quality again, extract DINOv2 embeddings again and retrain the models again.
Instead, it loads the CSV/JSON outputs generated by Dataset_Analysis_Analysis.ipynb and displays the  results.


In [13]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

DATASET_DIR = Path(
    r"C:\Users\USER\Downloads\clienti-20260916T131257Z-1-001\clienti"
)

def load_csv(name):
    return pd.read_csv(DATASET_DIR / name)

def load_json(name):
    with open(DATASET_DIR / name, "r") as f:
        return json.load(f)


## 1. Dataset size and labels

These runs show how many images were found, how many survived duplicate removal, and how the labels are distributed.


In [14]:
duplicate_summary = load_csv("duplicate_summary.csv")
label_status_counts = load_csv("label_status_counts.csv")
bbch_distribution = load_csv("bbch_distribution.csv")
other_labels = load_csv("other_raw_labels.csv")

print("Dataset summary")
display(duplicate_summary)

print("\nLabel status")
display(label_status_counts)

print("\nBBCH distribution")
display(bbch_distribution)

print("\nOther / unrecognized raw labels")
display(other_labels)


Dataset summary


,Unnamed: 0,quantity
0,initial_rows,1016
1,unique_paths,508
2,unique_exact_images,499
3,exact_duplicates_removed,517



Label status


,label_status,images
0,valid_bbch,381
1,other,91
2,NC,27



BBCH distribution


,bbch,images
0,0.0,6
1,1.0,1
2,3.0,2
3,4.0,1
4,5.0,3
5,6.0,4
6,10.0,2
7,11.0,4
8,12.0,12
9,13.0,22



Other / unrecognized raw labels


,raw_label,images
0,NaN,51
1,5,5
2,9,4
3,0,2
4,vigneto,2
5,incManBia,2
6,1,1
7,3,1
8,7,1
9,65,1


### What this section tells us

Initial rows are the images discovered from the dataset directory.
Unique paths removes repeated records pointing to the same physical file.
Unique exact images additionally removes files with identical SHA256 hashes.
The BBCH table shows which phenological stages are represented and how many images belong to each stage.
The other table shows labels that were not recognized as one of the explicitly defined BBCH values or NC.


## 2. Image quality and sharpness

The next runs summarize resolution, brightness/blur-related measurements, and the images with the lowest combined sharpness rank.


In [15]:
sharpness_statistics = load_csv("sharpness_statistics.csv")
lowest_sharpness = load_csv("lowest_sharpness_images.csv")

print("Image-quality / sharpness statistics")
display(sharpness_statistics)

print("\nImages with the lowest overall sharpness rank")
display(lowest_sharpness[
    [
        "filename", "bbch", "blur_score",
        "laplacian_var", "tenengrad",
        "edge_density", "overall_sharpness_rank"
    ]
])


Image-quality / sharpness statistics


,Unnamed: 0,blur_score,laplacian_var,tenengrad,edge_density
0,count,498.000000,498.000000,498.000000,498.000000
1,mean,317.286924,317.286924,5327.976471,0.042906
2,std,660.865386,660.865386,7874.424314,0.056933
3,min,3.287000,3.287000,67.406463,0.000000
4,25%,70.878615,70.878615,1138.219207,0.007856
5,50%,119.178086,119.178086,2422.286999,0.020847
6,75%,253.144047,253.144047,5579.941818,0.049859
7,max,7196.248253,7196.248253,51898.729870,0.337314



Images with the lowest overall sharpness rank


,filename,bbch,blur_score,laplacian_var,tenengrad,edge_density,overall_sharpness_rank
0,20260518_203856_NC.jpg,NaN,3.702045,3.702045,67.406463,0.000000,0.005020
1,20220413_170558_05.jpg,5.0,36.037242,36.037242,122.814705,0.000000,0.024431
2,20260324_112311_06.jpg,6.0,39.492786,39.492786,110.759024,0.000000,0.029786
3,20220316_174208_00.jpg,0.0,3.287000,3.287000,253.540556,0.000076,0.030790
4,20260519_57(2).jpg,57.0,34.566618,34.566618,161.496571,0.000046,0.031459
5,20220419_160609.jpg,NaN,7.071497,7.071497,228.429662,0.000305,0.034137
6,20240314_125612_00.jpg,0.0,38.415936,38.415936,178.262583,0.000000,0.034471
7,20260512_NC(2).jpg,NaN,48.927591,48.927591,163.187933,0.000000,0.037149
8,20260512_57(3).jpg,57.0,46.789642,46.789642,207.151698,0.000009,0.046854
9,20230606_121211_61.jpg,61.0,51.617879,51.617879,168.851744,0.000004,0.047523


In [16]:
blur_by_stage = load_csv("blur_by_bbch_stage.csv")

print("Blur statistics by BBCH stage")
display(blur_by_stage)


Blur statistics by BBCH stage


,bbch,count,mean,median,min,max
0,0.0,6,2436.786512,610.493400,3.287000,7196.248253
1,1.0,1,85.485291,85.485291,85.485291,85.485291
2,3.0,2,54.908777,54.908777,36.221091,73.596463
3,4.0,1,1026.427268,1026.427268,1026.427268,1026.427268
4,5.0,3,52.811757,51.709097,36.037242,70.688930
5,6.0,4,446.017479,330.700727,39.492786,1083.175678
6,10.0,2,903.346044,903.346044,104.945650,1701.746439
7,11.0,4,258.266488,134.663425,52.568502,711.170601
8,12.0,12,177.247422,108.511726,17.117401,827.941882
9,13.0,22,516.924105,150.137935,32.924730,4490.058961


### What this section tells us

The sharpness analysis uses three measurements:
1.Laplacian variance
2.Tenengrad
3.Edge density
The combined sharpness rank is based on the percentile rank of those three measurements. The table is therefore useful for identifying images that consistently receive low sharpness measurements.


## 3. BBCH stages and collection dates

These tables show when images were collected and which BBCH stages occurred on each date.


In [17]:
date_summary = load_csv("bbch_date_summary_long.csv")
year_stage_summary = load_csv("bbch_stage_summary_by_year.csv")
date_stage_counts = load_csv("bbch_stage_counts_by_date.csv")

print("Images by collection date and BBCH")
display(date_summary)

print("\nBBCH summary by year")
display(year_stage_summary)

print("\nBBCH stages by collection date")
display(date_stage_counts)


Images by collection date and BBCH


,date_dt,bbch,images
0,2021-08-27,83.0,1
1,2021-08-27,85.0,1
2,2021-09-30,89.0,1
3,2021-10-24,92.0,2
4,2022-03-16,0.0,1
...,...,...,...
127,2026-07-07,77.0,3
128,2026-07-18,79.0,3
129,2026-07-23,81.0,7
130,2026-08-04,85.0,2



BBCH summary by year


,year,bbch,images,first_date,last_date
0,2021.0,83.0,1,2021-08-27,2021-08-27
1,2021.0,85.0,1,2021-08-27,2021-08-27
2,2021.0,89.0,1,2021-09-30,2021-09-30
3,2021.0,92.0,2,2021-10-24,2021-10-24
4,2022.0,0.0,2,2022-03-16,2022-03-18
...,...,...,...,...,...
73,2026.0,77.0,14,2026-06-08,2026-07-07
74,2026.0,79.0,3,2026-07-18,2026-07-18
75,2026.0,81.0,7,2026-07-23,2026-07-23
76,2026.0,85.0,2,2026-08-04,2026-08-04



BBCH stages by collection date


,date_dt,0.0,1.0,3.0,4.0,5.0,6.0,10.0,11.0,12.0,...,77.0,79.0,81.0,83.0,85.0,87.0,89.0,91.0,92.0,95.0
0,2021-08-27,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
1,2021-09-30,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,2021-10-24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0
3,2022-03-16,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2022-03-18,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2026-07-07,0,0,0,0,0,0,0,0,0,...,3,0,0,0,0,0,0,0,0,0
80,2026-07-18,0,0,0,0,0,0,0,0,0,...,0,3,0,0,0,0,0,0,0,0
81,2026-07-23,0,0,0,0,0,0,0,0,0,...,0,0,7,0,0,0,0,0,0,0
82,2026-08-04,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0



The date analysis is important because images from the same collection date can be visually similar and can share environmental conditions. The date tables make it possible to see how the dataset covers the growing season.


## 4. Broader growth-stage groups

The original BBCH values are also grouped into the broader categories used by the classification experiment.


In [18]:
stage_counts = load_csv("growth_stage_balance.csv")

print("Growth-stage distribution")
display(stage_counts)


Growth-stage distribution


,growth_stage,images,percentage
0,fruit_development,155,40.68
1,flowering,91,23.88
2,early_development,77,20.21
3,ripening,58,15.22


The grouping used by the analysis code is:

BBCH 00–19
BBCH 51–69 
BBCH 71–79
BBCH 81–95


## 5. DINOv2 baseline

The following results come from DINOv2 embeddings followed by a balanced logistic-regression classifier.


In [19]:
baseline_metrics = load_json("dinov2_baseline_metrics.json")
baseline_confusion = load_csv("dinov2_growth_stage_confusion_matrix.csv")

print("DINOv2 baseline metrics")
display(pd.DataFrame([baseline_metrics]))

print("\nConfusion matrix")
display(baseline_confusion)


DINOv2 baseline metrics


,accuracy,balanced_accuracy
0,0.883117,0.87276



Confusion matrix


,Unnamed: 0,early_development,flowering,fruit_development,ripening
0,early_development,16,0,0,0
1,flowering,4,13,1,0
2,fruit_development,0,0,29,2
3,ripening,1,0,1,10


Accuracy measures the fraction of validation images classified correctly.
Balanced accuracy gives equal importance to the classes by averaging recall across classes. The confusion matrix shows where predictions and true growth-stage labels agree or differ.


## 6. Repeated random splits

Five stratified random splits were used to examine how much the measured performance changes across different train/validation partitions.


In [20]:
repeated_splits = load_csv("dinov2_repeated_split_results.csv")

print("Results for the five random splits")
display(repeated_splits)

print("Mean accuracy:", repeated_splits["accuracy"].mean())
print("Accuracy standard deviation:", repeated_splits["accuracy"].std())
print(
    "Mean balanced accuracy:",
    repeated_splits["balanced_accuracy"].mean()
)
print(
    "Balanced-accuracy standard deviation:",
    repeated_splits["balanced_accuracy"].std()
)


Results for the five random splits


,split,accuracy,balanced_accuracy
0,1,0.883117,0.872760
1,2,0.896104,0.882560
2,3,0.935065,0.913810
3,4,0.857143,0.840390
4,5,0.909091,0.891241


Mean accuracy: 0.8961038961038961
Accuracy standard deviation: 0.029039843863633653
Mean balanced accuracy: 0.880152329749104
Balanced-accuracy standard deviation: 0.02691346621804961


## 7. Date-held-out evaluation

This evaluation keeps complete collection dates together and uses the latest dates as validation data. This tests performance under a different distribution from a simple random split.


In [21]:
date_metrics = load_json("dinov2_date_heldout_metrics.json")
date_confusion = load_csv(
    "dinov2_date_heldout_confusion_matrix.csv"
)

print("Date-held-out metrics")
display(pd.DataFrame([date_metrics]))

print("\nDate-held-out confusion matrix")
display(date_confusion)


Date-held-out metrics


,accuracy,balanced_accuracy,training_images,validation_images
0,0.868421,0.778893,242,114



Date-held-out confusion matrix


,Unnamed: 0,early_development,flowering,fruit_development,ripening
0,early_development,0,0,0,0
1,flowering,0,16,7,0
2,fruit_development,0,4,74,0
3,ripening,0,0,4,9


## 8. One date held out from each growth stage

This experiment holds out the latest available date for each broader growth-stage group.


In [22]:
balanced_metrics = load_json(
    "dinov2_balanced_date_heldout_metrics.json"
)
balanced_confusion = load_csv(
    "dinov2_balanced_date_confusion_matrix.csv"
)
balanced_train = load_csv(
    "growth_stage_balanced_date_train.csv"
)
balanced_validation = load_csv(
    "growth_stage_balanced_date_validation.csv"
)

print("Balanced date-held-out metrics")
display(pd.DataFrame([balanced_metrics]))

print("\nBalanced date-held-out confusion matrix")
display(balanced_confusion)

print("\nValidation growth-stage distribution")
display(
    balanced_validation["growth_stage"]
    .value_counts()
    .rename_axis("growth_stage")
    .reset_index(name="images")
)


Balanced date-held-out metrics


,accuracy,balanced_accuracy,training_images,validation_images
0,0.27027,0.532609,344,37



Balanced date-held-out confusion matrix


,Unnamed: 0,early_development,flowering,fruit_development,ripening
0,early_development,3,20,0,0
1,flowering,0,0,7,0
2,fruit_development,0,0,3,0
3,ripening,0,0,0,4



Validation growth-stage distribution


,growth_stage,images
0,early_development,23
1,flowering,7
2,ripening,4
3,fruit_development,3


## 9. Image information versus date information

These results compare image embeddings only, date features only ,image embeddings plus date features
They are evaluated under both the random split and the phase-balanced date-held-out split.


In [23]:
comparison = load_csv("dinov2_image_date_comparison.csv")
late_fusion = load_csv(
    "dinov2_image_dominant_late_fusion.csv"
)

print("Image/date feature comparison")
display(comparison)

print("\nLate-fusion results")
display(late_fusion)


Image/date feature comparison


,split,model,accuracy,balanced_accuracy
0,random,image_only,0.883117,0.879704
1,random,date_only,0.883117,0.863463
2,random,image_plus_date,0.883117,0.879704
3,phase_balanced_date_heldout,image_only,0.270270,0.532609
4,phase_balanced_date_heldout,date_only,0.810811,0.750000
5,phase_balanced_date_heldout,image_plus_date,0.270270,0.532609



Late-fusion results


,split,image_weight,date_weight,accuracy,balanced_accuracy
0,random,0.9,0.1,0.883117,0.879704
1,random,0.8,0.2,0.883117,0.879704
2,random,0.7,0.3,0.883117,0.879704
3,phase_balanced_date_heldout,0.9,0.1,0.270270,0.532609
4,phase_balanced_date_heldout,0.8,0.2,0.270270,0.532609
5,phase_balanced_date_heldout,0.7,0.3,0.270270,0.532609


The purpose here was descriptive and it just shows how classification performance changes when the model is given visual information, seasonal/date information, or both. 
The late-fusion table additionally shows the tested weights for image probabilities and date probabilities.


## 10. Early phenological stages — BBCH 00–19

This is the subset relevant to the early vegetative phases.


In [24]:
early_resolution = load_csv(
    "early_bbch_00_19_resolution_statistics.csv"
)
early_unique_resolutions = load_csv(
    "early_bbch_00_19_unique_resolutions.csv"
)
early_blur = load_csv(
    "early_bbch_00_19_blur_statistics.csv"
)
early_by_stage = load_csv(
    "early_bbch_00_19_by_stage.csv"
)
early_details = load_csv(
    "early_bbch_00_19_image_details.csv"
)

print("Resolution statistics for BBCH 00–19")
display(early_resolution)

print("\nMost common resolutions")
display(early_unique_resolutions)

print("\nBlur statistics for BBCH 00–19")
display(early_blur)

print("\nBBCH 00–19 by stage")
display(early_by_stage)

print("\nImage-level details")
display(early_details)


Resolution statistics for BBCH 00–19


,Unnamed: 0,width,height
0,count,77.000000,77.000000
1,mean,4059.623377,3202.077922
2,std,1491.550333,902.921527
3,min,479.000000,385.000000
4,25%,4032.000000,3024.000000
5,50%,4624.000000,3468.000000
6,75%,4624.000000,3468.000000
7,max,5712.000000,4284.000000



Most common resolutions


,width,height,images
0,4624.0,3468.0,36
1,5712.0,4284.0,14
2,4032.0,3024.0,10
3,1500.0,2000.0,7
4,1080.0,1920.0,3
5,770.0,385.0,2
6,2080.0,3188.0,2
7,479.0,510.0,1
8,1920.0,1080.0,1
9,4000.0,3000.0,1



Blur statistics for BBCH 00–19


,Unnamed: 0,blur_score
0,count,77.000000
1,mean,471.683185
2,std,1190.085447
3,min,3.287000
4,25%,52.099913
5,50%,100.969946
6,75%,310.524504
7,max,7196.248253



BBCH 00–19 by stage


,bbch,images,min_width,median_width,min_height,median_height,median_blur,min_blur
0,0.0,6,1080.0,4624.0,1920.0,3468.0,610.493400,3.287000
1,1.0,1,4624.0,4624.0,3468.0,3468.0,85.485291,85.485291
2,3.0,2,4624.0,4624.0,3468.0,3468.0,54.908777,36.221091
3,4.0,1,4624.0,4624.0,3468.0,3468.0,1026.427268,1026.427268
4,5.0,3,4624.0,4624.0,3468.0,3468.0,51.709097,36.037242
5,6.0,4,4000.0,4624.0,3000.0,3468.0,330.700727,39.492786
6,10.0,2,4624.0,4624.0,3468.0,3468.0,903.346044,104.945650
7,11.0,4,770.0,3352.0,385.0,3328.0,134.663425,52.568502
8,12.0,12,770.0,4032.0,385.0,3024.0,108.511726,17.117401
9,13.0,22,1080.0,4624.0,1080.0,3468.0,150.137935,32.924730



Image-level details


,filename,bbch,width,height,blur_score,quality_flag
0,20260208_00.jpeg,0.0,1080.0,1920.0,7196.248253,OK
1,20260328_00.jpeg,0.0,1080.0,1920.0,6163.245348,OK
2,20230110_122603_00.jpg,0.0,4624.0,3468.0,36.951667,VERY_BLURRY
3,20220316_174208_00.jpg,0.0,4624.0,3468.0,3.287000,VERY_BLURRY
4,20240314_125612_00.jpg,0.0,4624.0,3468.0,38.415936,VERY_BLURRY
...,...,...,...,...,...,...
72,20260407_115904_15.jpeg,15.0,4032.0,3024.0,87.421946,BLURRY
73,20260413_135856_15.jpg,15.0,4624.0,3468.0,66.136238,BLURRY
74,20260413_135928_15.jpg,15.0,4624.0,3468.0,63.573958,BLURRY
75,20260407_115904_15(1).jpeg,15.0,5712.0,4284.0,23.804641,VERY_BLURRY
